> PostgreSQL 14+ practice. Run `sql/00_sample_data.sql` first, then `SET search_path TO sql_practice, public;` in this connection. Run cells in order in the same SQL session. See README.md for setup and expected results. These are SQL cells; the original Python notebook metadata did not make them executable as Python.


## Eliminating Key Lookups via Covering Indexes

In [ ]:
-- Create an index to support the following access pattern; indexes do not change payment values.
-- Equality column first, range column second, non-filtered payload in INCLUDE
CREATE INDEX IF NOT EXISTS ix_orders_cust_date_covering 
ON fact_orders (customer_id, order_date) 
INCLUDE (total_amount, currency_code);

## Single-column index

In [ ]:
-- Retrieve matching rows; compare EXPLAIN (ANALYZE, BUFFERS) before and after indexing.
SELECT *
FROM payment_transactions
WHERE customer_id = 201;

### Create an index

In [ ]:
-- Create an index to support the following access pattern; indexes do not change payment values.
CREATE INDEX IF NOT EXISTS idx_transactions_customer
ON payment_transactions (customer_id);

## Composite index (multiple columns)

In [ ]:
-- Retrieve matching rows; compare EXPLAIN (ANALYZE, BUFFERS) before and after indexing.
SELECT *
FROM payment_transactions
WHERE customer_id = 201
    AND status = 'Success';

### create the composite index

In [ ]:
-- Create an index to support the following access pattern; indexes do not change payment values.
CREATE INDEX IF NOT EXISTS idx_customer_status
ON payment_transactions (customer_id, status);

## Indexing for incremental data ingestion

In [ ]:
-- Retrieve matching rows; compare EXPLAIN (ANALYZE, BUFFERS) before and after indexing.
SELECT *
FROM payment_transactions
WHERE updated_at > '2026-11-26 10:00:00'
    AND updated_at <= '2026-11-26 11:00:00';

### Create the index

In [ ]:
-- Create an index to support the following access pattern; indexes do not change payment values.
CREATE INDEX IF NOT EXISTS idx_transactions_updated_at
ON payment_transactions (updated_at);

## Indexing for JOIN optimization

### Business requirement: Retrieve transactions alongside customer information.

In [ ]:
-- Retrieve matching rows; compare EXPLAIN (ANALYZE, BUFFERS) before and after indexing.
SELECT
    c.customer_name,
    t.transaction_id,
    t.amount
FROM customers c
INNER JOIN payment_transactions t
    ON c.customer_id = t.customer_id
WHERE c.customer_id = 201;

### The relevant index is:

In [ ]:
-- Create an index to support the following access pattern; indexes do not change payment values.
CREATE INDEX IF NOT EXISTS idx_transactions_customer
ON payment_transactions (customer_id);

In [ ]:
-- Inspect the indexes created in the current practice schema.
SELECT indexname, indexdef FROM pg_indexes
WHERE schemaname = current_schema()
ORDER BY indexname;